<a href="https://colab.research.google.com/github/10dimensions/gnc-toolbox/blob/main/pointing_error.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
import numpy as np

In [9]:
def normalize(vector):
    """Returns a unit vector (magnitude of 1) in the same direction."""
    norm = np.linalg.norm(vector)
    if norm == 0:
        return vector
    return vector / norm

In [10]:
def calculate_camera_pointing_error():
    print("--- GNC Pointing Utility: Earth Observation Camera ---\n")

    # =========================================================================
    # 1. NAVIGATION: Define positions in the ECI Frame (in kilometers)
    # =========================================================================
    # Spacecraft is on the X-axis, Target (hurricane) is on the Y-axis
    r_sc_eci = np.array([7000.0, 0.0, 0.0])
    r_target_eci = np.array([0.0, 7000.0, 0.0])

    # =========================================================================
    # 2. GUIDANCE: Calculate the desired pointing vector in ECI
    # =========================================================================
    # Vector from Spacecraft to Target
    v_desired_eci = r_target_eci - r_sc_eci
    v_desired_eci_norm = normalize(v_desired_eci)

    print(f"1. Desired Pointing Vector (ECI): {np.round(v_desired_eci_norm, 3)}")

    # =========================================================================
    # 3. ATTITUDE: Define the Direction Cosine Matrix (DCM)
    # =========================================================================
    # Let's assume the spacecraft is currently pitched up by 45 degrees
    # around the Y-axis. We construct the C_eci_to_body DCM manually.
    theta = np.radians(45.0) # 45 degree pitch

    # Rotation matrix around the Y-axis
    C_eci_to_body = np.array([
        [ np.cos(theta), 0, np.sin(theta)],
        [ 0,             1, 0            ],
        [-np.sin(theta), 0, np.cos(theta)]
    ])

    print(f"2. Spacecraft Attitude: Pitched up 45 deg around Y-axis.\n")

    # =========================================================================
    # 4. TRANSFORMATION: Convert the desired vector to the Body Frame
    # =========================================================================
    # Matrix multiplication: v_body = C_eci_to_body * v_eci
    v_desired_body = C_eci_to_body @ v_desired_eci_norm

    print(f"3. Desired Pointing Vector (Body): {np.round(v_desired_body, 3)}")

    # =========================================================================
    # 5. CONTROL: Calculate the Pointing Error
    # =========================================================================
    # The camera is mounted on the -Z axis of the Body frame.
    # Therefore, the "ideal" camera boresight vector in the Body frame is [0, 0, -1]
    v_camera_boresight = np.array([0.0, 0.0, -1.0])

    # Calculate the angular error between where the camera IS pointing
    # and where it NEEDS to point.
    dot_product = np.dot(v_camera_boresight, v_desired_body)
    # Clip to [-1, 1] to prevent math domain errors due to floating point drift
    angle_error_rad = np.arccos(np.clip(dot_product, -1.0, 1.0))
    angle_error_deg = np.degrees(angle_error_rad)

    # The cross product gives the axis of rotation needed to correct the error
    rotation_axis = np.cross(v_camera_boresight, v_desired_body)

    print(f"4. Camera Boresight (Body):      {v_camera_boresight}")
    print(f"5. Pointing Error (Angle):       {np.round(angle_error_deg, 2)} degrees")
    print(f"6. Required Rotation Axis (Body): {np.round(normalize(rotation_axis), 3)}")
    print("\n-> Control System will now use this error to calculate reaction wheel torques!")

In [11]:
if __name__ == "__main__":
    calculate_camera_pointing_error()

--- GNC Pointing Utility: Earth Observation Camera ---

1. Desired Pointing Vector (ECI): [-0.707  0.707  0.   ]
2. Spacecraft Attitude: Pitched up 45 deg around Y-axis.

3. Desired Pointing Vector (Body): [-0.5    0.707  0.5  ]
4. Camera Boresight (Body):      [ 0.  0. -1.]
5. Pointing Error (Angle):       120.0 degrees
6. Required Rotation Axis (Body): [0.816 0.577 0.   ]

-> Control System will now use this error to calculate reaction wheel torques!
